# TrustVis — Semantic Region Attribution for Deepfake Detectors (corrected pipeline, camera-ready)

**What changed vs. the submitted notebook** (each fix is marked `# FIX n` in the code):

| # | Problem in submitted notebook | Effect on the paper | Fix |
|---|---|---|---|
| 1 | `pip install pytorch-grad-cam` (not a PyPI name) | install cell errors | `pip install grad-cam` |
| 2 | Grad-CAM++ returned **all-zero maps** for many images (ViT ≈55 %, MobileNetV2 ≈45 %); `compute_region_attribution` turned them into 0 % for every region and they were averaged in | Table 1 rows for ViT/MobileNetV2 sum to 45.3 / 54.8 %; non-facial % and R = −0.89 are biased | empty maps are detected, counted, reported, and excluded from means |
| 3 | Target layers: MobileNetV2/EfficientNet used `blocks[-1]` (linear bottleneck, no activation); Inception-ResNet-v2 matched the `'resnet'` branch and fell back to a pre-BN conv; Xception/VGG used pre-activation convs | negative CAMs → empty maps; layer choice differs from what the paper implies | post-activation layers: `bn2`, `conv2d_7b`, `act4`, last VGG ReLU (legacy layers still available) |
| 4 | `test_dataset.dataset.transform = test_transform` changed the **shared** dataset, so training ran without the flip augmentation | silent | separate dataset objects |
| 5 | Random **frame-level** split: frames of the same video in train and test; fakes = first *n* paths in sorted order | FF++ accuracy inflated; fake methods unevenly sampled | optional video-level split (`SPLIT_BY_VIDEO`) with round-robin frame sampling |
| 6 | Paper Eq. (3) = cosine of mean vectors; code = mean of per-image cosines | Table 3 does not follow from Table 1 | both reported, clearly named |
| 7 | No area-normalised attribution (reviewer request) | — | `E_k = R_k / area_k` reported next to `R_k` |
| 8 | Integrated Gradients never run in this notebook (Sec. 4.5 claims r = 0.91) | claim not reproducible from this code | IG cell added (black baseline, as stated in paper) |
| 9 | Paper says 5 epochs; code trains 6 | text/code mismatch | config kept at 6 — **update the paper** |
| 10 | Face parsing run once per model per image; CAM object rebuilt per image | 8× slower; noisy logs | parse once per dataset; batched CAM |
| 11 | No statistics on the n = 8 correlation | over-claim | Pearson/Spearman, p, bootstrap CI, leave-one-out |

**Fastest camera-ready path:** if you still have the submitted checkpoints, set `SPLIT_BY_VIDEO = False` (reuses `*_ffplus.pth`, no retraining) and keep `LEGACY_TARGET_LAYERS = False`. Run `LEGACY_TARGET_LAYERS = True` once as well if you want the before/after numbers.

In [ ]:
# FIX 1: the PyPI package is `grad-cam` (it is imported as `pytorch_grad_cam`)
!pip install -q grad-cam captum timm gdown

## Configuration

In [ ]:
import os, sys, gc, json, re, random, warnings, subprocess
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from captum.attr import IntegratedGradients

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

FFPLUS_ROOT = "/kaggle/input/datasets/adham7elmy/faceforencispp-extracted-frames"
CELEB_DF_ROOT = "/kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2/Test"
WORK_DIR = "/kaggle/working/trustvis_results"

IMG_SIZE = 224
BATCH_SIZE = 24
TRAIN_EPOCHS = 6            # FIX 9: the submitted paper says 5 epochs — the code trains 6. Update the paper.
LEARNING_RATE = 1e-4
NUM_WORKERS = 2
RANDOM_SEED = 42

N_TRAIN = 4000              # 2,000 real + 2,000 fake
N_TEST_FF = 2000            # 1,000 real + 1,000 fake
N_TEST_CROSS = 1000         # 500 real + 500 fake (Celeb-DF v2)

SPLIT_BY_VIDEO = True       # FIX 5. False = submitted frame-level split (reuses the old *_ffplus.pth checkpoints)
VIDEO_TEST_FRACTION = 0.2   # share of videos held out for the FF++ test pool
LEGACY_TARGET_LAYERS = False  # FIX 3. True = exactly the Grad-CAM++ layers used for the submitted paper
EXPLAIN_TARGET = 'pred'     # 'pred' = explain the predicted class (submitted protocol); 'label' = ground-truth class
EXPLAIN_BATCH = 16
SAL_STORE = 56              # resolution at which saliency maps are kept for pixel-level agreement analysis
EMPTY_EPS = 1e-8            # a map with total mass <= EMPTY_EPS is "empty"

RUN_IG = True               # FIX 8
N_IG_IMAGES = 500
N_IG_STEPS = 50
IG_INTERNAL_BATCH = 10      # keeps IG within T4 memory
IG_BASELINE = 'black'       # 'black' (as the paper states) or 'zero' (= ImageNet-mean grey, what the old code did)

PRETRAINED = True
MODEL_NAMES = ['xception', 'efficientnet_b4', 'vit_base_patch16_224', 'resnet50',
               'vgg16', 'vgg19', 'mobilenetv2_100', 'inception_resnet_v2']

SPLIT_TAG = 'videosplit' if SPLIT_BY_VIDEO else 'ffplus'   # 'ffplus' = name of the submitted checkpoints
LAYER_TAG = 'legacylayers' if LEGACY_TARGET_LAYERS else 'fixedlayers'
RUN_TAG = f"{SPLIT_TAG}_{LAYER_TAG}_{EXPLAIN_TARGET}"
CKPT_DIR = f"{WORK_DIR}/checkpoints"
OUT_DIR = f"{WORK_DIR}/{RUN_TAG}"
for d in [WORK_DIR, CKPT_DIR, OUT_DIR, f"{OUT_DIR}/figures", f"{OUT_DIR}/tables", f"{OUT_DIR}/records"]:
    os.makedirs(d, exist_ok=True)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False


set_seed(RANDOM_SEED)
print("Run tag:", RUN_TAG)

## Face parser (BiSeNet, CelebAMask-HQ, 19 classes)

In [ ]:
import gdown

FACE_PARSING_WEIGHTS_PATH = f"{WORK_DIR}/79999_iter.pth"
FACE_PARSING_REPO_PATH = f"{WORK_DIR}/face-parsing.PyTorch"

if not os.path.exists(FACE_PARSING_REPO_PATH):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/zllrunning/face-parsing.PyTorch.git',
                    FACE_PARSING_REPO_PATH], check=True)
if not os.path.exists(FACE_PARSING_WEIGHTS_PATH):
    gdown.download(id="154JgKpzCPW82qINcVieuPH3fZ2e0P812", output=FACE_PARSING_WEIGHTS_PATH, quiet=False)

sys.path.insert(0, FACE_PARSING_REPO_PATH)
from model import BiSeNet   # no silent random-weight fallback: if this fails, the parsing would be meaningless

face_parser = BiSeNet(n_classes=19)
face_parser.load_state_dict(torch.load(FACE_PARSING_WEIGHTS_PATH, map_location='cpu'))
face_parser = face_parser.to(DEVICE).eval()
print("Face parser loaded.")

## Region definition and per-image region statistics

In [ ]:
# CelebAMask-HQ label ids produced by face-parsing.PyTorch (79999_iter.pth):
#  0 background  1 skin  2 l_brow  3 r_brow  4 l_eye  5 r_eye  6 eyeglasses  7 l_ear  8 r_ear
#  9 earring  10 nose  11 mouth(inner)  12 u_lip  13 l_lip  14 neck  15 necklace  16 cloth  17 hair  18 hat
# NOTE for the paper: there is no beard/facial-hair class, so facial hair is labelled mostly as skin;
# ears, neck, clothes, hat and accessories are in 'background'; eyeglasses are in 'eyes'.
REGION_MAP = {
    'eyes': [2, 3, 4, 5, 6],
    'nose': [10],
    'mouth': [11, 12, 13],
    'skin': [1],
    'hair': [17],
    'background': [0, 7, 8, 9, 14, 15, 16, 18],
}
REGION_NAMES = ['eyes', 'nose', 'mouth', 'skin', 'hair', 'background']
K_REGIONS = len(REGION_NAMES)
_ids = sorted(i for v in REGION_MAP.values() for i in v)
assert _ids == list(range(19)), "regions must partition the 19 parser classes"
CLASS_TO_REGION = np.zeros(19, dtype=np.uint8)
for k, name in enumerate(REGION_NAMES):
    CLASS_TO_REGION[REGION_MAP[name]] = k

_PARSE_MEAN = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 3, 1, 1)
_PARSE_STD = torch.tensor(IMAGENET_STD, device=DEVICE).view(1, 3, 1, 1)


@torch.no_grad()
def parse_faces(images_uint8):
    """images_uint8: (B,H,W,3) RGB uint8 -> region index maps (B,H,W) uint8 in 0..5."""
    B, H, W, _ = images_uint8.shape
    x = torch.from_numpy(images_uint8).to(DEVICE).permute(0, 3, 1, 2).float() / 255.0
    x = F.interpolate(x, size=(512, 512), mode='bilinear', align_corners=False)
    x = (x - _PARSE_MEAN) / _PARSE_STD
    out = face_parser(x)[0]
    labels = out.argmax(1).cpu().numpy().astype(np.uint8)
    res = np.stack([cv2.resize(l, (W, H), interpolation=cv2.INTER_NEAREST) for l in labels])
    return CLASS_TO_REGION[res]


def region_stats(saliency, region_idx):
    """Returns (R, area): R = % of total saliency mass per region (None if the map is empty),
    area = % of image pixels per region.  Regions partition the image, so a valid R sums to 100."""
    s = np.asarray(saliency, dtype=np.float64)
    flat_idx = region_idx.ravel()
    area = np.bincount(flat_idx, minlength=K_REGIONS) / flat_idx.size * 100.0
    total = s.sum()
    if not np.isfinite(total) or total <= EMPTY_EPS:
        return None, area           # FIX 2: never turn an empty map into "0 % everywhere"
    R = np.bincount(flat_idx, weights=s.ravel(), minlength=K_REGIONS) / total * 100.0
    return R, area

print("Region helpers ready.")

## Data: sample lists, video-level split, datasets

In [ ]:
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
REAL_NAMES = ['real', '0_real', 'original', '0', 'Real', 'ORIGINAL']
FAKE_NAMES = ['fake', '1_fake', 'manipulated', '1', 'Fake', 'FAKE']


def find_class_dirs(root):
    for base in [root] + [os.path.join(root, d) for d in sorted(os.listdir(root)) if os.path.isdir(os.path.join(root, d))]:
        r = next((os.path.join(base, n) for n in REAL_NAMES if os.path.isdir(os.path.join(base, n))), None)
        f = next((os.path.join(base, n) for n in FAKE_NAMES if os.path.isdir(os.path.join(base, n))), None)
        if r and f:
            return r, f
    raise ValueError(f"Could not find real/fake sub-directories in {root}")


def list_images(d):
    out = []
    for dp, _, files in os.walk(d):
        out += [os.path.join(dp, f) for f in files if f.endswith(IMG_EXTS)]   # case-sensitive, like the submitted glob
    return sorted(out)


_FRAME_SUFFIX = re.compile(r'[_\-\s.]*(?:frame|f|img)?[_\-\s]*\d+$', re.IGNORECASE)


def video_id(path, class_dir):
    """Video key = sub-folder path + file stem with the trailing frame index removed.
    Handles 'real/000/0012.png' -> '000', 'fake/Deepfakes/000_003_frame12.jpg' -> 'Deepfakes/000_003'.
    CHECK the printed examples: if your file names do not encode the video, edit this function."""
    rel = os.path.relpath(path, class_dir)
    folder, fname = os.path.split(rel)
    stem = os.path.splitext(fname)[0]
    vid = _FRAME_SUFFIX.sub('', stem)
    return os.path.join(folder, vid) if vid else (folder or stem)


def group_by_video(paths, class_dir):
    g = {}
    for p in paths:
        g.setdefault(video_id(p, class_dir), []).append(p)
    return g


def describe_groups(name, groups, n_images):
    sizes = np.array([len(v) for v in groups.values()])
    print(f"  {name}: {n_images} frames in {len(groups)} videos (frames/video median {np.median(sizes):.0f}, "
          f"min {sizes.min()}, max {sizes.max()}); examples: {list(groups)[:3]}")
    if len(groups) < 20 or len(groups) == n_images:
        print(f"  WARNING: video grouping for {name} looks wrong (too few groups, or one frame per group). "
              f"Inspect the file names and edit video_id() before trusting a video-level split.")


def sample_round_robin(groups, vids, n, rng):
    """Take frames one video at a time so that n frames come from as many videos as possible."""
    pools = {v: list(rng.permutation(groups[v])) for v in vids}
    order = list(vids)
    rng.shuffle(order)
    out = []
    while len(out) < n and any(pools[v] for v in order):
        for v in order:
            if pools[v]:
                out.append(pools[v].pop())
                if len(out) >= n:
                    break
    if len(out) < n:
        print(f"  note: only {len(out)} frames available (requested {n})")
    return out


def build_ffpp_splits():
    real_dir, fake_dir = find_class_dirs(FFPLUS_ROOT)
    real, fake = list_images(real_dir), list_images(fake_dir)
    print(f"FF++: {len(real)} real, {len(fake)} fake frames  ({real_dir} | {fake_dir})")
    if not SPLIT_BY_VIDEO:
        # Submitted protocol, reproduced exactly: balance by taking the first n sorted paths of each class,
        # shuffle, then a random FRAME-level split (frames of one video can be in both train and test).
        n = min(len(real), len(fake))
        samples = [(p, 0) for p in real[:n]] + [(p, 1) for p in fake[:n]]
        random.seed(RANDOM_SEED)
        random.shuffle(samples)
        idx = list(range(len(samples)))
        random.shuffle(idx)
        train = [samples[i] for i in idx[:N_TRAIN]]
        test = [samples[i] for i in idx[N_TRAIN:N_TRAIN + N_TEST_FF]]
        return train, test
    rng = np.random.default_rng(RANDOM_SEED)
    train, test = [], []
    for label, cdir, paths in [(0, real_dir, real), (1, fake_dir, fake)]:
        groups = group_by_video(paths, cdir)
        describe_groups('real' if label == 0 else 'fake', groups, len(paths))
        vids = sorted(groups)
        rng.shuffle(vids)
        n_test_v = max(1, int(round(VIDEO_TEST_FRACTION * len(vids))))
        test_v, train_v = vids[:n_test_v], vids[n_test_v:]
        tr = sample_round_robin(groups, train_v, N_TRAIN // 2, rng)
        te = sample_round_robin(groups, test_v, N_TEST_FF // 2, rng)
        assert not {video_id(p, cdir) for p in tr} & {video_id(p, cdir) for p in te}, "video leakage!"
        train += [(p, label) for p in tr]
        test += [(p, label) for p in te]
    rng.shuffle(train)
    rng.shuffle(test)
    return train, test


def build_celebdf_test():
    real_dir, fake_dir = find_class_dirs(CELEB_DF_ROOT)
    real, fake = list_images(real_dir), list_images(fake_dir)
    print(f"Celeb-DF: {len(real)} real, {len(fake)} fake frames")
    if not SPLIT_BY_VIDEO:
        n = min(len(real), len(fake), N_TEST_CROSS // 2)     # submitted protocol: first n sorted paths
        samples = [(p, 0) for p in real[:n]] + [(p, 1) for p in fake[:n]]
        random.Random(RANDOM_SEED).shuffle(samples)
        return samples
    rng = np.random.default_rng(RANDOM_SEED + 1)
    out = []
    for label, cdir, paths in [(0, real_dir, real), (1, fake_dir, fake)]:
        groups = group_by_video(paths, cdir)
        describe_groups('celeb real' if label == 0 else 'celeb fake', groups, len(paths))
        out += [(p, label) for p in sample_round_robin(groups, sorted(groups), N_TEST_CROSS // 2, rng)]
    rng.shuffle(out)
    return out


train_transform = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(), T.ToTensor(),
                             T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
test_transform = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])


class FrameDataset(Dataset):
    """FIX 4: each split owns its own dataset object (and therefore its own transform).
    with_raw=True also returns the resized uint8 image used by the face parser."""
    def __init__(self, samples, transform, with_raw=False):
        self.samples, self.transform, self.with_raw = list(samples), transform, with_raw

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert('RGB')
        x = self.transform(img)
        if not self.with_raw:
            return x, label
        raw = np.asarray(img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.uint8)
        return x, torch.from_numpy(raw.copy()), label, i


ff_train_samples, ff_test_samples = build_ffpp_splits()
celeb_samples = build_celebdf_test()
print(f"train {len(ff_train_samples)} | FF++ test {len(ff_test_samples)} | Celeb-DF {len(celeb_samples)}")
for nm, s in [('train', ff_train_samples), ('ff_test', ff_test_samples), ('celeb_df', celeb_samples)]:
    pd.DataFrame(s, columns=['path', 'label']).to_csv(f"{OUT_DIR}/tables/split_{nm}.csv", index=False)

train_loader = DataLoader(FrameDataset(ff_train_samples, train_transform), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)

## Detectors and Grad-CAM++ target layers

In [ ]:
def build_detector(name):
    return timm.create_model(name, pretrained=PRETRAINED, num_classes=2)


def reshape_transform_vit(tensor, height=14, width=14):
    tensor = tensor[:, 1:, :]                       # drop CLS token
    tensor = tensor.reshape(tensor.size(0), height, width, tensor.size(-1))
    return tensor.permute(0, 3, 1, 2)


def legacy_target_layers(model, name):
    """Exactly what the submitted notebook selected (kept for before/after comparison)."""
    n = name.lower()
    if 'vit' in n:
        return [model.blocks[-1].norm1]
    if 'xception' in n:
        return [model.conv4]                        # SeparableConv2d, before bn4/act4
    if 'efficientnet' in n or 'mobilenet' in n:
        return [model.blocks[-1]]                   # linear bottleneck output (no activation)
    if 'resnet' in n and hasattr(model, 'layer4'):
        return [model.layer4[-1]]
    if 'vgg' in n:
        return [[m for m in model.features if isinstance(m, nn.Conv2d)][-1]]   # pre-ReLU conv
    # inception_resnet_v2 fell through the 'resnet' branch to the generic fallback: last Conv2d
    return [[m for m in model.modules() if isinstance(m, nn.Conv2d)][-1]]      # conv2d_7b.conv (pre-BN)


def target_layers_for(model, name):
    """FIX 3: last spatial feature map AFTER its non-linearity, i.e. the tensor global pooling actually sees."""
    if LEGACY_TARGET_LAYERS:
        return legacy_target_layers(model, name)
    n = name.lower()
    if n.startswith('vit'):
        return [model.blocks[-1].norm1]             # standard choice for ViT Grad-CAM; empty maps are logged
    if 'inception_resnet' in n:                     # must be tested BEFORE 'resnet'
        return [model.conv2d_7b]                    # 1x1 conv + BN + ReLU, 1536 ch
    if 'xception' in n:
        return [model.act4]
    if 'efficientnet' in n or 'mobilenet' in n:
        return [model.bn2]                          # conv_head -> BN -> SiLU/ReLU6, 1792 / 1280 ch
    if 'resnet' in n:
        return [model.layer4[-1]]
    if 'vgg' in n:
        relu = [m for m in model.features if isinstance(m, nn.ReLU)][-1]
        relu.inplace = False
        return [relu]
    raise ValueError(f"No target layer rule for {name}")


for _n in MODEL_NAMES:
    _m = timm.create_model(_n, pretrained=False, num_classes=2)
    _new = target_layers_for(_m, _n)[0]
    _old = legacy_target_layers(_m, _n)[0]
    _lookup = {mod: nm for nm, mod in _m.named_modules()}
    print(f"{_n:24s} layer used: {_lookup[_new]:28s} (submitted: {_lookup[_old]})")
    del _m

## Training (skipped when a checkpoint for this split exists)

In [ ]:
def ckpt_path(name):
    return f"{CKPT_DIR}/{name}_{SPLIT_TAG}.pth"


def train_or_load(name):
    model = build_detector(name).to(DEVICE)
    path = ckpt_path(name)
    if os.path.exists(path):
        print(f"  loading {path}")
        model.load_state_dict(torch.load(path, map_location=DEVICE))
        return model.eval()

    set_seed(RANDOM_SEED)
    model.train()
    # Normalisation layers are frozen (eval mode, no grad) — state this in the paper's training protocol.
    for module in model.modules():
        if isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d, nn.LayerNorm)):
            module.eval()
            for p in module.parameters():
                p.requires_grad = False
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_EPOCHS * len(train_loader))
    for epoch in range(TRAIN_EPOCHS):
        tot, correct, loss_sum = 0, 0, 0.0
        for x, y in tqdm(train_loader, desc=f"{name} epoch {epoch + 1}/{TRAIN_EPOCHS}", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            scheduler.step()
            loss_sum += loss.item()
            correct += (out.argmax(1) == y).sum().item()
            tot += y.size(0)
        print(f"  {name} epoch {epoch + 1}: loss {loss_sum / len(train_loader):.4f}, train acc {100 * correct / tot:.2f}%")
    torch.save(model.state_dict(), path)
    return model.eval()


for name in MODEL_NAMES:
    if not os.path.exists(ckpt_path(name)):
        m = train_or_load(name)
        del m
        gc.collect()
        torch.cuda.empty_cache()
print("All checkpoints present:", all(os.path.exists(ckpt_path(n)) for n in MODEL_NAMES))

## Explanations: parse each dataset once, then batched Grad-CAM++ per detector

In [ ]:
def make_loader(samples):
    return DataLoader(FrameDataset(samples, test_transform, with_raw=True), batch_size=EXPLAIN_BATCH,
                      shuffle=False, num_workers=NUM_WORKERS)


def parse_dataset(samples, tag):
    cache = f"{WORK_DIR}/parse_{tag}_{SPLIT_TAG}.npz"
    paths = np.array([p for p, _ in samples])
    if os.path.exists(cache):
        z = np.load(cache, allow_pickle=False)
        if np.array_equal(z['paths'], paths):
            return {'region_idx': z['region_idx'], 'area': z['area'], 'paths': paths}
    idx_maps = np.zeros((len(samples), IMG_SIZE, IMG_SIZE), np.uint8)
    for _, raw, _, ids in tqdm(make_loader(samples), desc=f"parsing {tag}"):
        idx_maps[ids.numpy()] = parse_faces(raw.numpy())
    area = np.stack([np.bincount(m.ravel(), minlength=K_REGIONS) / m.size * 100 for m in idx_maps])
    np.savez_compressed(cache, region_idx=idx_maps, area=area, paths=paths)
    return {'region_idx': idx_maps, 'area': area, 'paths': paths}


def _empty_record(samples):
    N = len(samples)
    return {'paths': np.array([p for p, _ in samples]), 'label': np.array([l for _, l in samples]),
            'pred': np.zeros(N, int), 'prob_fake': np.zeros(N, np.float32),
            'R': np.full((N, K_REGIONS), np.nan), 'valid': np.zeros(N, bool),
            'sal': np.zeros((N, SAL_STORE, SAL_STORE), np.float16)}


def explain_gradcam(name, samples, parse, tag):
    out_file = f"{OUT_DIR}/records/{tag}_{name}_gradcampp.npz"
    if os.path.exists(out_file):
        z = np.load(out_file, allow_pickle=False)
        return {k: z[k] for k in z.files}
    model = train_or_load(name)
    reshape = reshape_transform_vit if name.lower().startswith('vit') else None
    cam = GradCAMPlusPlus(model=model, target_layers=target_layers_for(model, name), reshape_transform=reshape)
    rec = _empty_record(samples)
    try:
        for x, _, y, ids in tqdm(make_loader(samples), desc=f"{name} | {tag}"):
            x = x.to(DEVICE)
            with torch.no_grad():
                probs = F.softmax(model(x), 1)
            pred = probs.argmax(1)
            tgt = pred if EXPLAIN_TARGET == 'pred' else y.to(DEVICE)
            maps = cam(input_tensor=x, targets=[ClassifierOutputTarget(int(t)) for t in tgt])   # (B,H,W), [0,1]
            ids = ids.numpy()
            rec['pred'][ids] = pred.cpu().numpy()
            rec['prob_fake'][ids] = probs[:, 1].cpu().numpy()
            for j, i in enumerate(ids):
                R, _ = region_stats(maps[j], parse['region_idx'][i])
                if R is not None:
                    rec['R'][i], rec['valid'][i] = R, True
                rec['sal'][i] = cv2.resize(maps[j].astype(np.float32), (SAL_STORE, SAL_STORE), interpolation=cv2.INTER_AREA)
    finally:
        cam.activations_and_grads.release()     # remove hooks
        del model
        gc.collect()
        torch.cuda.empty_cache()
    np.savez_compressed(out_file, **rec)
    print(f"  {name} on {tag}: acc {100 * (rec['pred'] == rec['label']).mean():.2f}%, "
          f"empty maps {100 * (1 - rec['valid'].mean()):.1f}%")
    return rec


parses = {'ff_test': parse_dataset(ff_test_samples, 'ff_test'), 'celeb_df': parse_dataset(celeb_samples, 'celeb_df')}
records = {'ff_test': {}, 'celeb_df': {}}
for tag, samples in [('ff_test', ff_test_samples), ('celeb_df', celeb_samples)]:
    for name in MODEL_NAMES:
        records[tag][name] = explain_gradcam(name, samples, parses[tag], tag)

## Analysis (shared code — identical in `camera_ready_analysis.ipynb`)

In [ ]:
# ==================== ANALYSIS CORE (shared by both notebooks) ====================
# Every analysis works on one "record" per (dataset, model), a dict of aligned numpy arrays:
#   paths (N,)  label (N,)  pred (N,)  prob_fake (N,)
#   R      (N,6)  relative attribution in %, NaN rows where the saliency map was empty
#   valid  (N,)   False where Grad-CAM++/IG returned an all-zero map
#   sal    (N,s,s) float16 saliency map stored at low resolution (for spatial agreement)
# and one parse dict per dataset: region_idx (N,H,W) uint8 in 0..5, area (N,6) in %.
# Images are in the same order for every model of a dataset.
import numpy as np
import pandas as pd
from scipy import stats

REGION_NAMES = ['eyes', 'nose', 'mouth', 'skin', 'hair', 'background']
FACIAL_FEATURES = ['eyes', 'nose', 'mouth']
NON_FACIAL = ['hair', 'background']
PRETTY = {
    'xception': 'Xception', 'efficientnet_b4': 'EfficientNet-B4', 'vit_base_patch16_224': 'ViT-B/16',
    'resnet50': 'ResNet50', 'vgg16': 'VGG16', 'vgg19': 'VGG19', 'mobilenetv2_100': 'MobileNetV2',
    'inception_resnet_v2': 'Inception-ResNet-v2',
}
_RI = {r: i for i, r in enumerate(REGION_NAMES)}


def _idx(names):
    return [_RI[n] for n in names]


def _cos_rows(a, b):
    na, nb = np.linalg.norm(a, axis=1), np.linalg.norm(b, axis=1)
    return (a * b).sum(1) / (na * nb)


def _cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def _auc(label, score):
    try:
        from sklearn.metrics import roc_auc_score
        if len(np.unique(label)) < 2:
            return np.nan
        return 100.0 * roc_auc_score(label, score)
    except Exception:
        return np.nan


def subset_mask(rec, which):
    """which: 'all' | 'fake' | 'real' | 'correct' | 'correct_fake' | 'wrong'."""
    lab, pred = rec['label'], rec['pred']
    return {
        'all': np.ones(len(lab), bool),
        'fake': lab == 1,
        'real': lab == 0,
        'correct': pred == lab,
        'correct_fake': (pred == lab) & (lab == 1),
        'wrong': pred != lab,
    }[which]


def summarize_regions(records, parse, which='all'):
    """Table 1 replacement.  R_k: mean relative attribution over images with a non-empty map
    (each row sums to 100).  A_k: mean area share of the region.  E_k = R_k / A_k is the
    area-normalised attribution (1.0 = the region gets exactly its pixel share; >1 = it gets
    more saliency per pixel than average).  E_k is proportional to the paper's Eq. (1) A_{d,k}."""
    rows = []
    for m, r in records.items():
        sel = subset_mask(r, which)
        v = r['valid'] & sel
        n_sel = int(sel.sum())
        row = {'model': m, 'subset': which, 'n_images': n_sel, 'n_valid_maps': int(v.sum()),
               'empty_map_rate_%': 100.0 * (1 - v.sum() / max(n_sel, 1)),
               'accuracy_%': 100.0 * float((r['pred'][sel] == r['label'][sel]).mean()) if n_sel else np.nan,
               'auc_%': _auc(r['label'][sel], r['prob_fake'][sel]) if n_sel else np.nan}
        if v.sum() == 0:
            rows.append(row)
            continue
        Rm = np.nanmean(r['R'][v], axis=0)
        # what the submitted Table 1 reported: empty maps counted as 0 % everywhere
        legacy = np.where(r['valid'][:, None], np.nan_to_num(r['R']), 0.0)[sel].mean(0)
        Am = parse['area'][v].mean(0)
        E = Rm / np.maximum(Am, 1e-9)
        for k, name in enumerate(REGION_NAMES):
            row[f'R_{name}'] = Rm[k]
            row[f'A_{name}'] = Am[k]
            row[f'E_{name}'] = E[k]
            row[f'legacyR_{name}'] = legacy[k]
        row['R_sum'] = Rm.sum()
        row['legacyR_sum'] = legacy.sum()
        row['R_non_facial'] = Rm[_idx(NON_FACIAL)].sum()
        row['R_features'] = Rm[_idx(FACIAL_FEATURES)].sum()
        row['A_non_facial'] = Am[_idx(NON_FACIAL)].sum()
        row['E_non_facial'] = row['R_non_facial'] / max(row['A_non_facial'], 1e-9)
        row['A_features'] = Am[_idx(FACIAL_FEATURES)].sum()
        row['E_features'] = row['R_features'] / max(row['A_features'], 1e-9)
        row['legacyR_non_facial'] = legacy[_idx(NON_FACIAL)].sum()
        rows.append(row)
    return pd.DataFrame(rows)


def pairwise_tables(records, top_frac=0.2):
    """SAC under both definitions + spatial agreement + error overlap, for all model pairs.
    SAC_per_image : mean over images of cos(R_d1(x), R_d2(x))   <- what the submitted Table 3 contains
    SAC_global    : cos(mean_x R_d1, mean_x R_d2)                <- what the submitted Eq. (3) states
    spatial_r     : mean per-image Pearson r between the two saliency maps (pixel level)
    top_iou       : mean per-image IoU of the top-`top_frac` most salient pixels
    err_jaccard   : |E1 & E2| / |E1 | E2| over misclassified images
    kappa_correct : Cohen's kappa between the two models' correct/incorrect vectors."""
    models = list(records)
    out = []
    for i, m1 in enumerate(models):
        for m2 in models[i + 1:]:
            r1, r2 = records[m1], records[m2]
            assert np.array_equal(r1['paths'], r2['paths']), 'records must be image-aligned'
            both = r1['valid'] & r2['valid']
            cs = _cos_rows(r1['R'][both], r2['R'][both])
            g1 = np.nanmean(r1['R'][r1['valid']], 0)
            g2 = np.nanmean(r2['R'][r2['valid']], 0)
            s1 = r1['sal'][both].reshape(both.sum(), -1).astype(np.float32)
            s2 = r2['sal'][both].reshape(both.sum(), -1).astype(np.float32)
            pr, ious = [], []
            k = max(1, int(round(top_frac * s1.shape[1]))) if len(s1) else 1
            for a, b in zip(s1, s2):
                if a.std() > 0 and b.std() > 0:
                    pr.append(np.corrcoef(a, b)[0, 1])
                ta = np.zeros(a.shape, bool); ta[np.argpartition(-a, k - 1)[:k]] = True
                tb = np.zeros(b.shape, bool); tb[np.argpartition(-b, k - 1)[:k]] = True
                ious.append((ta & tb).sum() / (ta | tb).sum())
            c1 = r1['pred'] == r1['label']
            c2 = r2['pred'] == r2['label']
            e1, e2 = ~c1, ~c2
            union = (e1 | e2).sum()
            try:
                from sklearn.metrics import cohen_kappa_score
                kappa = cohen_kappa_score(c1, c2) if (len(np.unique(c1)) > 1 or len(np.unique(c2)) > 1) else np.nan
            except Exception:
                kappa = np.nan
            out.append({
                'model_1': m1, 'model_2': m2, 'n_both_valid': int(both.sum()),
                'SAC_per_image': float(cs.mean()) if len(cs) else np.nan,
                'SAC_per_image_std': float(cs.std()) if len(cs) else np.nan,
                'SAC_global': _cos(g1, g2),
                'top_region_agree_%': 100.0 * float((r1['R'][both].argmax(1) == r2['R'][both].argmax(1)).mean()) if both.any() else np.nan,
                'spatial_r': float(np.mean(pr)) if pr else np.nan,
                f'top{int(top_frac * 100)}_iou': float(np.mean(ious)) if ious else np.nan,
                'err_jaccard': float((e1 & e2).sum() / union) if union else np.nan,
                'both_wrong_%': 100.0 * float((e1 & e2).mean()),
                'kappa_correct': kappa,
            })
    return pd.DataFrame(out)


def pair_matrix(df_pairs, col, models):
    M = np.ones((len(models), len(models)))
    for _, row in df_pairs.iterrows():
        i, j = models.index(row['model_1']), models.index(row['model_2'])
        M[i, j] = M[j, i] = row[col]
    return pd.DataFrame(M, index=[PRETTY.get(m, m) for m in models], columns=[PRETTY.get(m, m) for m in models])


def corr_report(x, y, n_boot=10000, seed=0, label=''):
    """Pearson + Spearman + bootstrap 95% CI + leave-one-detector-out range (n is tiny: be honest)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    pr, pp = stats.pearsonr(x, y)
    sr, sp = stats.spearmanr(x, y)
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        i = rng.integers(0, len(x), len(x))
        if len(np.unique(i)) < 3 or np.std(x[i]) == 0 or np.std(y[i]) == 0:
            continue
        boots.append(np.corrcoef(x[i], y[i])[0, 1])
    loo = [np.corrcoef(np.delete(x, j), np.delete(y, j))[0, 1] for j in range(len(x))]
    return {'what': label, 'n': len(x), 'pearson_r': pr, 'pearson_p': pp, 'spearman_rho': sr, 'spearman_p': sp,
            'boot95_lo': np.percentile(boots, 2.5), 'boot95_hi': np.percentile(boots, 97.5),
            'loo_min': min(loo), 'loo_max': max(loo)}


def generalisation_table(df_src, df_tgt):
    a = df_src[['model', 'accuracy_%', 'auc_%', 'R_non_facial', 'E_non_facial', 'legacyR_non_facial', 'R_skin', 'E_skin', 'empty_map_rate_%']]
    b = df_tgt[['model', 'accuracy_%', 'auc_%']]
    d = a.merge(b, on='model', suffixes=('_src', '_tgt'))
    d['acc_drop_pp'] = d['accuracy_%_src'] - d['accuracy_%_tgt']
    d['auc_drop_pp'] = d['auc_%_src'] - d['auc_%_tgt']
    return d


def correlation_suite(gen):
    rows = []
    for xcol, lab in [('legacyR_non_facial', 'non-facial R as submitted (empty maps = 0)'),
                      ('R_non_facial', 'non-facial R, empty maps excluded'),
                      ('E_non_facial', 'non-facial area-normalised E'),
                      ('R_skin', 'skin R, empty maps excluded')]:
        for ycol in ['acc_drop_pp', 'auc_drop_pp']:
            if gen[ycol].notna().all():
                rows.append(corr_report(gen[xcol], gen[ycol], label=f'{lab} vs {ycol}'))
    return pd.DataFrame(rows)


def latex_table1(df, caption_note=''):
    lines = [r'\begin{tabular}{l' + 'c' * 9 + '}', r'\toprule',
             r'Detector & Eyes & Nose & Mouth & Skin & Hair & Bg & Non-facial & Features & Empty (\%) \\',
             r'\midrule']
    for _, r in df.iterrows():
        cells = [f"{r[f'R_{k}']:.1f} ({r[f'E_{k}']:.2f})" for k in REGION_NAMES]
        cells += [f"{r['R_non_facial']:.1f} ({r['E_non_facial']:.2f})", f"{r['R_features']:.1f} ({r['E_features']:.2f})",
                  f"{r['empty_map_rate_%']:.1f}"]
        lines.append(PRETTY.get(r['model'], r['model']) + ' & ' + ' & '.join(cells) + r' \\')
    lines += [r'\bottomrule', r'\end{tabular}']
    return '\n'.join(lines)


def latex_pairs(df_pairs, cols=('SAC_per_image', 'SAC_global', 'spatial_r', 'top20_iou', 'err_jaccard')):
    head = 'Detector pair & ' + ' & '.join(c.replace('_', r'\_') for c in cols) + r' \\'
    lines = [r'\begin{tabular}{l' + 'c' * len(cols) + '}', r'\toprule', head, r'\midrule']
    for _, r in df_pairs.iterrows():
        lines.append(f"{PRETTY.get(r['model_1'], r['model_1'])} vs. {PRETTY.get(r['model_2'], r['model_2'])} & "
                     + ' & '.join(f'{r[c]:.3f}' for c in cols) + r' \\')
    lines += [r'\bottomrule', r'\end{tabular}']
    return '\n'.join(lines)

In [ ]:
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 80)
TAB = f"{OUT_DIR}/tables"

df_ff = summarize_regions(records['ff_test'], parses['ff_test'], 'all')
df_cd = summarize_regions(records['celeb_df'], parses['celeb_df'], 'all')
df_ff.to_csv(f"{TAB}/table1_ff_all.csv", index=False)
df_cd.to_csv(f"{TAB}/regions_celebdf_all.csv", index=False)

show = ['model', 'accuracy_%', 'auc_%', 'empty_map_rate_%', 'legacyR_sum', 'R_sum'] + \
       [f'R_{k}' for k in REGION_NAMES] + ['R_non_facial', 'R_features']
print("FF++ — relative attribution R (%), empty maps excluded:")
print(df_ff[show].round(2).to_string(index=False))
print("\nFF++ — area share A (%) and area-normalised attribution E = R/A (1.0 = uniform):")
print(df_ff[['model'] + [f'A_{k}' for k in REGION_NAMES] + [f'E_{k}' for k in REGION_NAMES] + ['E_non_facial', 'E_features']].round(2).to_string(index=False))

# Protocol sensitivity (reviewer rYMW): same table restricted to subsets
for which in ['correct_fake', 'fake', 'real']:
    d = summarize_regions(records['ff_test'], parses['ff_test'], which)
    d.to_csv(f"{TAB}/table1_ff_{which}.csv", index=False)
    print(f"\nFF++ subset = {which}:")
    print(d[['model', 'n_images', 'empty_map_rate_%'] + [f'R_{k}' for k in REGION_NAMES] + ['R_non_facial', 'E_non_facial']].round(2).to_string(index=False))

with open(f"{TAB}/table1_ff_all.tex", 'w') as f:
    f.write(latex_table1(df_ff))

In [ ]:
gen = generalisation_table(df_ff, df_cd)
gen.to_csv(f"{TAB}/table2_generalisation.csv", index=False)
print(gen[['model', 'accuracy_%_src', 'accuracy_%_tgt', 'acc_drop_pp', 'auc_%_src', 'auc_%_tgt', 'auc_drop_pp',
           'legacyR_non_facial', 'R_non_facial', 'E_non_facial']].round(2).to_string(index=False))
corr = correlation_suite(gen)
corr.to_csv(f"{TAB}/correlations.csv", index=False)
print("\nCorrelations across the 8 detectors (n = 8 — report as exploratory):")
print(corr.round(3).to_string(index=False))

In [ ]:
pairs_ff = pairwise_tables(records['ff_test'])
pairs_cd = pairwise_tables(records['celeb_df'])
pairs_ff.to_csv(f"{TAB}/table3_pairs_ff.csv", index=False)
pairs_cd.to_csv(f"{TAB}/pairs_celebdf.csv", index=False)
print("FF++ pairwise (SAC both definitions, pixel-level agreement, error overlap):")
print(pairs_ff.round(3).to_string(index=False))
print("\nCeleb-DF error overlap:")
print(pairs_cd[['model_1', 'model_2', 'SAC_per_image', 'spatial_r', 'err_jaccard', 'both_wrong_%', 'kappa_correct']].round(3).to_string(index=False))
with open(f"{TAB}/table3_pairs_ff.tex", 'w') as f:
    f.write(latex_pairs(pairs_ff))

# Does high SAC imply pixel-level / failure agreement?  (reviewer rYMW, meta-review)
print("\nAcross the 28 pairs (FF++ SAC vs. pixel agreement; FF++ SAC vs. Celeb-DF error overlap):")
print(pd.DataFrame([corr_report(pairs_ff['SAC_per_image'], pairs_ff['spatial_r'], label='SAC vs spatial_r (FF++)'),
                    corr_report(pairs_ff['SAC_per_image'], pairs_cd['err_jaccard'], label='SAC(FF++) vs err_jaccard(Celeb-DF)')]).round(3).to_string(index=False))

## Integrated Gradients validation (Sec. 4.5)

In [ ]:
def ig_baseline(x):
    if IG_BASELINE == 'black':     # a black image in normalised space
        mean = torch.tensor(IMAGENET_MEAN, device=x.device).view(1, 3, 1, 1)
        std = torch.tensor(IMAGENET_STD, device=x.device).view(1, 3, 1, 1)
        return ((0.0 - mean) / std).expand_as(x).contiguous()
    return torch.zeros_like(x)     # all-zero tensor = ImageNet-mean grey image (old behaviour)


def explain_ig(name, samples, parse, tag):
    out_file = f"{OUT_DIR}/records/{tag}_{name}_ig.npz"
    if os.path.exists(out_file):
        z = np.load(out_file, allow_pickle=False)
        return {k: z[k] for k in z.files}
    model = train_or_load(name)
    ig = IntegratedGradients(model)
    rec = _empty_record(samples)
    for x, _, y, ids in tqdm(make_loader(samples), desc=f"IG {name}"):
        x = x.to(DEVICE)
        with torch.no_grad():
            probs = F.softmax(model(x), 1)
        pred = probs.argmax(1)
        tgt = pred if EXPLAIN_TARGET == 'pred' else y.to(DEVICE)
        ids = ids.numpy()
        rec['pred'][ids] = pred.cpu().numpy()
        rec['prob_fake'][ids] = probs[:, 1].cpu().numpy()
        for j, i in enumerate(ids):
            a = ig.attribute(x[j:j + 1], baselines=ig_baseline(x[j:j + 1]), target=int(tgt[j]), n_steps=N_IG_STEPS,
                             method='gausslegendre', internal_batch_size=IG_INTERNAL_BATCH)
            s = a.squeeze(0).abs().mean(0).detach().cpu().numpy()   # |attribution|, averaged over channels
            R, _ = region_stats(s, parse['region_idx'][i])
            if R is not None:
                rec['R'][i], rec['valid'][i] = R, True
            smax = s.max() if s.max() > 0 else 1.0
            rec['sal'][i] = cv2.resize((s / smax).astype(np.float32), (SAL_STORE, SAL_STORE), interpolation=cv2.INTER_AREA)
    del model, ig
    gc.collect()
    torch.cuda.empty_cache()
    np.savez_compressed(out_file, **rec)
    return rec


if RUN_IG:
    ig_samples = ff_test_samples[:N_IG_IMAGES]
    ig_parse = {'region_idx': parses['ff_test']['region_idx'][:N_IG_IMAGES], 'area': parses['ff_test']['area'][:N_IG_IMAGES]}
    rows, gc_means, ig_means = [], [], []
    for name in MODEL_NAMES:
        ig_rec = explain_ig(name, ig_samples, ig_parse, 'ff_test_igsubset')
        gc_rec = {k: (v[:N_IG_IMAGES] if hasattr(v, '__len__') else v) for k, v in records['ff_test'][name].items()}
        both = ig_rec['valid'] & gc_rec['valid']
        g_mean = np.nanmean(gc_rec['R'][both], 0)
        i_mean = np.nanmean(ig_rec['R'][both], 0)
        gc_means.append(g_mean)
        ig_means.append(i_mean)
        rows.append({'model': name, 'n': int(both.sum()),
                     'per_image_cos(GC++,IG)': float(_cos_rows(gc_rec['R'][both], ig_rec['R'][both]).mean()),
                     'cos(mean GC++, mean IG)': _cos(g_mean, i_mean),
                     'GC++ non-facial': g_mean[[4, 5]].sum(), 'IG non-facial': i_mean[[4, 5]].sum()})
    df_ig = pd.DataFrame(rows)
    gm, im = np.concatenate(gc_means), np.concatenate(ig_means)
    pooled = corr_report(gm, im, label='pooled 8x6 mean-R vectors, GC++ vs IG')
    rank = corr_report(df_ig['GC++ non-facial'], df_ig['IG non-facial'], label='non-facial % across detectors, GC++ vs IG')
    df_ig.to_csv(f"{TAB}/ig_validation.csv", index=False)
    print(df_ig.round(3).to_string(index=False))
    print(pd.DataFrame([pooled, rank]).round(3).to_string(index=False))
    print("NOTE: the pooled r is dominated by 'skin vs small regions' and is high almost by construction; "
          "the per-image cosine and the across-detector non-facial ranking are the informative numbers.")

## Figures

In [ ]:
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.3)
FIG = f"{OUT_DIR}/figures"
REGION_COLOURS = np.array([[230, 25, 75], [245, 130, 48], [255, 225, 25], [60, 180, 75], [0, 130, 200], [128, 128, 128]], np.uint8)

# Fig: region attribution (R) and area-normalised attribution (E)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
d = df_ff.set_index(df_ff['model'].map(PRETTY))
d[[f'R_{k}' for k in REGION_NAMES]].set_axis(REGION_NAMES, axis=1).plot.bar(ax=axes[0], width=0.8, colormap='tab10')
axes[0].set_ylabel('Relative attribution $R_{d,k}$ (%)')
d[[f'E_{k}' for k in REGION_NAMES]].set_axis(REGION_NAMES, axis=1).plot.bar(ax=axes[1], width=0.8, colormap='tab10', legend=False)
axes[1].axhline(1.0, color='k', ls='--', lw=1)
axes[1].set_ylabel('Area-normalised attribution $R_{d,k}/\\mathrm{area}_k$')
for a in axes:
    a.set_xlabel('')
    a.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(f"{FIG}/region_attribution_R_and_E.png", dpi=300, bbox_inches='tight')
plt.show()

# Fig 3: non-facial attribution vs accuracy drop (corrected values)
fig, ax = plt.subplots(figsize=(8, 6))
x, y = gen['R_non_facial'].values, gen['acc_drop_pp'].values
ax.scatter(x, y, s=120, edgecolors='k', zorder=3)
for xi, yi, m in zip(x, y, gen['model']):
    ax.annotate(PRETTY.get(m, m), (xi, yi), textcoords='offset points', xytext=(6, 4), fontsize=10)
z = np.polyfit(x, y, 1)
xl = np.linspace(x.min() - 2, x.max() + 2, 50)
c = corr_report(x, y)
ax.plot(xl, np.polyval(z, xl), 'k--', alpha=0.6, label=f"r = {c['pearson_r']:.2f} (p = {c['pearson_p']:.3f}, n = {len(x)})")
ax.set_xlabel('Non-facial attribution on FF++ (hair + background, %)')
ax.set_ylabel('Accuracy drop FF++ → Celeb-DF (pp)')
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG}/cross_dataset_drop.png", dpi=300, bbox_inches='tight')
plt.show()

# Fig 4: SAC heatmaps — per-image SAC, global SAC, pixel-level agreement
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for a, col, ttl in zip(axes, ['SAC_per_image', 'SAC_global', 'spatial_r'],
                       ['SAC (mean per-image cosine)', 'SAC of mean vectors (Eq. 3 as submitted)', 'Pixel-level Pearson r']):
    sns.heatmap(pair_matrix(pairs_ff, col, MODEL_NAMES), annot=True, fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1,
                square=True, ax=a, cbar=False)
    a.set_title(ttl)
plt.tight_layout()
plt.savefig(f"{FIG}/sac_heatmaps.png", dpi=300, bbox_inches='tight')
plt.show()


# Fig 2 / Fig 1 example: image, 6-region parsing (matches the method: eyes, nose, mouth, skin, hair, background), CAMs
def figure_examples(tag='ff_test', models=('xception', 'efficientnet_b4', 'vit_base_patch16_224', 'resnet50'), n=3):
    recs, parse = records[tag], parses[tag]
    ok = np.where(np.all([recs[m]['valid'] for m in models], axis=0) & (recs[models[0]]['label'] == 1))[0][:n]
    fig, axes = plt.subplots(len(ok), len(models) + 2, figsize=(3.2 * (len(models) + 2), 3.2 * len(ok)), squeeze=False)
    for r, i in enumerate(ok):
        img = np.asarray(Image.open(recs[models[0]]['paths'][i]).convert('RGB').resize((IMG_SIZE, IMG_SIZE)))
        axes[r, 0].imshow(img)
        axes[r, 1].imshow(REGION_COLOURS[parse['region_idx'][i]])
        for j, m in enumerate(models):
            s = cv2.resize(recs[m]['sal'][i].astype(np.float32), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
            axes[r, j + 2].imshow(img)
            axes[r, j + 2].imshow(s, cmap='jet', alpha=0.5)
            Rm = recs[m]['R'][i]
            axes[r, j + 2].set_xlabel(f"skin {Rm[3]:.0f}% | hair {Rm[4]:.0f}% | bg {Rm[5]:.0f}%", fontsize=9)
            if r == 0:
                axes[r, j + 2].set_title(PRETTY.get(m, m))
        if r == 0:
            axes[r, 0].set_title('Input (fake)')
            axes[r, 1].set_title('Parsing (6 regions)')
        for a in axes[r]:
            a.set_xticks([]); a.set_yticks([])
    handles = [plt.Rectangle((0, 0), 1, 1, color=REGION_COLOURS[k] / 255) for k in range(K_REGIONS)]
    fig.legend(handles, REGION_NAMES, loc='lower center', ncol=6, frameon=False)
    plt.tight_layout(rect=(0, 0.05, 1, 1))
    plt.savefig(f"{FIG}/saliency_examples.png", dpi=300, bbox_inches='tight')
    plt.show()


figure_examples()

## Summary file

In [ ]:
summary = {
    'run_tag': RUN_TAG, 'date': str(pd.Timestamp.now()),
    'config': {'img_size': IMG_SIZE, 'train_epochs': TRAIN_EPOCHS, 'lr': LEARNING_RATE, 'batch_size': BATCH_SIZE,
               'n_train': len(ff_train_samples), 'n_test_ff': len(ff_test_samples), 'n_test_cross': len(celeb_samples),
               'split_by_video': SPLIT_BY_VIDEO, 'legacy_target_layers': LEGACY_TARGET_LAYERS,
               'explain_target': EXPLAIN_TARGET, 'frozen_norm_layers': True, 'ig_baseline': IG_BASELINE,
               'ig_steps': N_IG_STEPS, 'n_ig_images': N_IG_IMAGES if RUN_IG else 0},
    'target_layers': {n: str(type(target_layers_for(timm.create_model(n, pretrained=False, num_classes=2), n)[0]).__name__)
                      for n in MODEL_NAMES},
}
with open(f"{OUT_DIR}/experiment_summary.json", 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print("Tables:", sorted(os.listdir(TAB)))
print("Figures:", sorted(os.listdir(FIG)))